In [8]:
from dotenv import load_dotenv
load_dotenv("../.env")  # load the repo-root .env so this works outside VS Code too
from decouple import config

In [9]:
assert config("ALPHA_VANTAGE_API_KEY", default=None, cast=str) is not None

In [10]:
ALPHA_VANTAGE_API_KEY = config("ALPHA_VANTAGE_API_KEY", default=None, cast=str)

In [20]:
import requests

# replace the "demo" apikey below with your own key from https://www.alphavantage.co/support/#api-key


api_key = ALPHA_VANTAGE_API_KEY

params = {
    "api_key": ALPHA_VANTAGE_API_KEY,
    "ticker": "AAPL",
    "function": "TIME_SERIES_DAILY"
}

url = 'https://www.alphavantage.co/query?function={function}&symbol={ticker}&apikey={api_key}'.format(
    **params
)
r = requests.get(url)
data = r.json()
data

{'Meta Data': {'1. Information': 'Daily Prices (open, high, low, close) and Volumes',
  '2. Symbol': 'AAPL',
  '3. Last Refreshed': '2026-10-19',
  '4. Output Size': 'Compact',
  '5. Time Zone': 'US/Eastern'},
 'Time Series (Daily)': {'2026-10-19': {'1. open': '340.7750',
   '2. high': '346.0000',
   '3. low': '340.1150',
   '4. close': '342.9400',
   '5. volume': '34573181'},
  '2026-10-16': {'1. open': '346.0850',
   '2. high': '347.8200',
   '3. low': '340.4600',
   '4. close': '343.4600',
   '5. volume': '59946368'},
  '2026-10-15': {'1. open': '346.9500',
   '2. high': '347.8500',
   '3. low': '341.6000',
   '4. close': '342.0000',
   '5. volume': '39767256'},
  '2026-10-14': {'1. open': '344.6050',
   '2. high': '347.0900',
   '3. low': '341.7200',
   '4. close': '342.9600',
   '5. volume': '50061053'},
  '2026-10-13': {'1. open': '344.6650',
   '2. high': '348.1499',
   '3. low': '340.7400',
   '4. close': '341.2700',
   '5. volume': '38983839'},
  '2026-10-12': {'1. open': '338

In [47]:
for x in data.keys():
    if not x.startswith("Meta Data"):
        dataset_key = x
        
dataset_key
value=data.get(dataset_key, None)
value

timestamp = list(value.keys())
timestamp

first_timestamp = timestamp[0]
first_timestamp

check_value_in_timestamp = value[first_timestamp]
check_value_in_timestamp


{'1. open': '340.7750',
 '2. high': '346.0000',
 '3. low': '340.1150',
 '4. close': '342.9400',
 '5. volume': '34573181'}

In [35]:
dataset_key = [x for x in list(data.keys()) if not x.lower() == "meta data"][0]
dataset_key


'Time Series (Daily)'

In [36]:
results = data[dataset_key] # dict
# for timestamp_str in results.keys():
#     print(timestamp_str, results.get(timestamp_str))

In [14]:
timestamp_str = list(results.keys())[0]
timestamp_str

'2026-10-19'

In [15]:
result_val = results.get(timestamp_str)
result_val

{'1. open': '340.7750',
 '2. high': '346.0000',
 '3. low': '340.1150',
 '4. close': '342.9400',
 '5. volume': '34573181'}

In [16]:
import pytz
from datetime import datetime
from decimal import Decimal

def transform_alpha_vantage_result(timestamp_str, result):
    # unix_timestamp = result.get('t') / 1000.0
    # utc_timestamp = datetime.fromtimestamp(unix_timestamp, tz=pytz.timezone('UTC'))
    timestamp_format = '%Y-%m-%d' 
    eastern = pytz.timezone("US/Eastern")
    utc = pytz.utc
    timestamp = eastern.localize(datetime.strptime(timestamp_str,timestamp_format).replace(hour=16)).astimezone(utc)
    return {
        'open_price': Decimal(result['1. open']),
        'close_price': Decimal(result['4. close']),
        'high_price': Decimal(result['2. high']),
        'low_price': Decimal(result['3. low']),
        'number_of_trades': None,
        'volume': int(result['5. volume']),
        'volume_weighted_average': None,
        'time': timestamp,
    }

In [17]:
transform_alpha_vantage_result(timestamp_str, result_val)

{'open_price': Decimal('340.7750'),
 'close_price': Decimal('342.9400'),
 'high_price': Decimal('346.0000'),
 'low_price': Decimal('340.1150'),
 'number_of_trades': None,
 'volume': 34573181,
 'volume_weighted_average': None,
 'time': datetime.datetime(2026, 10, 19, 20, 0, tzinfo=<UTC>)}

In [18]:
for timestamp_str in results.keys():
    new_data = transform_alpha_vantage_result(timestamp_str, results.get(timestamp_str))
    print(new_data)

{'open_price': Decimal('340.7750'), 'close_price': Decimal('342.9400'), 'high_price': Decimal('346.0000'), 'low_price': Decimal('340.1150'), 'number_of_trades': None, 'volume': 34573181, 'volume_weighted_average': None, 'time': datetime.datetime(2026, 10, 19, 20, 0, tzinfo=<UTC>)}
{'open_price': Decimal('346.0850'), 'close_price': Decimal('343.4600'), 'high_price': Decimal('347.8200'), 'low_price': Decimal('340.4600'), 'number_of_trades': None, 'volume': 59946368, 'volume_weighted_average': None, 'time': datetime.datetime(2026, 10, 16, 20, 0, tzinfo=<UTC>)}
{'open_price': Decimal('346.9500'), 'close_price': Decimal('342.0000'), 'high_price': Decimal('347.8500'), 'low_price': Decimal('341.6000'), 'number_of_trades': None, 'volume': 39767256, 'volume_weighted_average': None, 'time': datetime.datetime(2026, 10, 15, 20, 0, tzinfo=<UTC>)}
{'open_price': Decimal('344.6050'), 'close_price': Decimal('342.9600'), 'high_price': Decimal('347.0900'), 'low_price': Decimal('341.7200'), 'number_of_tr